In [ ]:
!pip install google-genai chromadb

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 705.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.

In [ ]:
import os
import chromadb
from google import genai

os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6JBzmOzYNhbRr8_eskdW0iwKavhzMq2Quefgvpjpi6L0w"

client = genai.Client()

print("Initializing ChromaDB vector database...")
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="artist_portfolio")
except:
    pass

collection = chroma_client.create_collection(name="artist_portfolio")

artworks = [
    {
        "id": "art_001",
        "title": "Whispers of the Valley",
        "medium": "Oil on Canvas",
        "year": "2023",
        "description": "The artwork 'Whispers of the Valley' is a 2023 oil painting. The core concept explores the coexistence of nature and industry. The warm, natural valley landscape is cut by cold-toned geometric shapes. The artist created this piece to reflect on modern urban expansion."
    },
    {
        "id": "art_002",
        "title": "The Observer's Rest",
        "medium": "Digital Painting",
        "year": "2024",
        "description": "The artwork 'The Observer's Rest' is a digital painting. The core concept: it depicts two cats named Snow and Lichard resting quietly in a sunlit server room, symbolizing the warmth, tranquility, and companionship brought by small lives amidst cold technology and massive data flows."
    },
    {
        "id": "art_003",
        "title": "Fragmented Memories",
        "medium": "Mixed Media",
        "year": "2025",
        "description": "The artwork 'Fragmented Memories' uses mixed media, including torn old newspapers and acrylic paint. This piece captures the passage of time and explores how the human brain automatically fills in the blanks of memories."
    }
]

for art in artworks:
    collection.add(
        documents=[art["description"]],
        metadatas=[{"title": art["title"], "medium": art["medium"], "year": art["year"]}],
        ids=[art["id"]]
    )

print("Exclusive gallery database created successfully!\n" + "="*50)

test_query = "Please describe in detail the core concept of the digital artwork 'The Observer's Rest'. Which two animals are mentioned in it?"

def run_zero_shot(query):
    """Experiment A: Traditional Zero-Shot LLM (no external background knowledge provided)"""
    print("\n[Experiment A: Zero-Shot LLM (Relying purely on model memory)]")
    print(f"User Query: {query}")
    print("Generating response...\n")

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=query
    )
    print(f"Generated Output:\n{response.text.strip()}")
    print("-" * 50)

def run_rag_system(query):
    """Experiment B: Retrieval-Augmented Generation (fetch data from DB first, then let LLM answer)"""
    print("\n[Experiment B: RAG System (Combining ChromaDB retrieval and Gemini generation)]")
    print(f"User Query: {query}")

    print("Retrieving relevant documents from the vector database...")
    results = collection.query(
        query_texts=[query],
        n_results=1
    )

    retrieved_context = results['documents'][0][0]
    retrieved_metadata = results['metadatas'][0][0]
    print(f"Retrieved Background Knowledge: ({retrieved_metadata['title']}) {retrieved_context}\n")

    print("Generating response using background knowledge...\n")
    rag_prompt = f"""
    You are a professional digital gallery guide. Please answer the user's question strictly based on the provided [Background Knowledge].
    If the background knowledge does not mention relevant information, please honestly answer "According to the current gallery data, I do not have this information." You must not fabricate information.

    [Background Knowledge]:
    {retrieved_context}

    [User Query]:
    {query}
    """

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=rag_prompt
    )
    print(f"Final Generated Output:\n{response.text.strip()}")
    print("=" * 50)

if __name__ == "__main__":
    run_zero_shot(test_query)
    run_rag_system(test_query)

Initializing ChromaDB vector database...
Exclusive gallery database created successfully!

[Experiment A: Zero-Shot LLM (Relying purely on model memory)]
User Query: Please describe in detail the core concept of the digital artwork 'The Observer's Rest'. Which two animals are mentioned in it?
Generating response...

Generated Output:
The core concept of the digital artwork 'The Observer's Rest' is a profound meditation on the **interplay between the organic and the digital, the act of observation, and the search for tranquility in an increasingly data-saturated world.**

This artwork typically presents a liminal space where hyperrealistic natural landscapes – perhaps a serene forest, a placid lake at dawn, or a rugged mountain vista – are subtly yet inextricably interwoven with ethereal, luminous digital elements. These digital intrusions might manifest as shimmering data streams, translucent geometric grids, pulsating network patterns, or even ghostly digital echoes of past events or 